# Roulette Analytics Lab

An independent Python reimplementation and extension of the mathematical specification in the MATH20062 Group 40 roulette report. The notebook uses the same tested domain functions as the dashboard and published tables.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from roulette_lab.analysis import AnalysisConfig, run_full_analysis
from roulette_lab.bets import expected_net_return
from roulette_lab.statistics import max_count_test

pd.options.display.float_format = "{:.6f}".format
print("Python 3.12 | shared package interfaces loaded")

Python 3.12 | shared package interfaces loaded


In [2]:
config = AnalysisConfig.fast_test()
bundle = run_full_analysis(config)
print(f"Seed: {config.seed}; tables: {len(bundle.table_names())}")

Seed: 20260912; tables: 7


## House Edge

For a stake of one unit, exact expected net return is $E[X]=pb-(1-p)$. The typed bet fixes the casino payout. A fair European straight-up bet therefore returns $-1/37$ in expectation, rather than zero.

In [3]:
bundle.house_edges[["rule", "bet", "expected_net_return", "house_edge"]]

,rule,bet,expected_net_return,house_edge
0,european,straight,-0.027027,0.027027
1,american,straight,-0.052632,0.052632
2,european_standard,red,-0.027027,0.027027
3,la_partage,red,-0.013514,0.013514
4,en_prison,red,-0.013514,0.013514


## Law of Large Numbers

The cumulative frequency of pocket 17 fluctuates at small samples and approaches the model probability as the number of spins grows.

In [4]:
bundle.lln_convergence.iloc[[0, 9, 99, -1]][["spin", "empirical_probability", "theoretical_probability", "absolute_error"]]

,spin,empirical_probability,theoretical_probability,absolute_error
0,1,0.000000,0.027027,0.027027
9,10,0.000000,0.027027,0.027027
99,100,0.020000,0.027027,0.007027
799,800,0.016250,0.027027,0.010777


## Bias Detection

A global Pearson test asks whether the whole wheel departs from fairness. The maximum-count test then corrects for choosing the hottest pocket after seeing the same data. The family-wise value is the relevant measure for that selected claim.

In [5]:
bundle.bias_tests[["dataset", "hottest_label", "asymptotic_p_value", "monte_carlo_global_p_value", "naive_p_value", "bonferroni_p_value", "familywise_p_value"]]

,dataset,hottest_label,asymptotic_p_value,monte_carlo_global_p_value,naive_p_value,bonferroni_p_value,familywise_p_value
0,unbiased,32,0.397832,0.379052,0.016397,0.606686,0.516209
1,biased,17,0.002893,0.009975,0.000000,0.000000,0.002494


## Kelly Criterion

For net odds $b$ and estimated win probability $p$, full Kelly is $f^* = \max(0,(bp-(1-p))/b)$. A straight-up 35:1 bet needs $p>1/36$ for a positive allocation. Fractional Kelly reduces model and path risk but does not create an edge.

In [6]:
bundle.kelly_sensitivity.loc[bundle.kelly_sensitivity["pocket_probability"].between(1/37, 0.04)].head(8)

,pocket_probability,break_even_probability,full_kelly,half_kelly,quarter_kelly
3,0.027027,0.027778,0.000000,0.000000,0.000000
4,0.027500,0.027778,0.000000,0.000000,0.000000
5,0.027778,0.027778,0.000000,0.000000,0.000000
6,0.030000,0.027778,0.002286,0.001143,0.000571
7,0.032500,0.027778,0.004857,0.002429,0.001214
8,0.035000,0.027778,0.007429,0.003714,0.001857
9,0.037500,0.027778,0.010000,0.005000,0.002500
10,0.040000,0.027778,0.012571,0.006286,0.003143


## Bankroll Risk

Terminal averages alone hide dispersion. The comparison reports medians, simulation intervals, loss probability, ruin probability and maximum drawdown under one declared biased-wheel scenario with table constraints.

In [7]:
bundle.strategy_risk[["strategy", "terminal_mean", "terminal_median", "probability_of_loss", "probability_of_ruin", "expected_maximum_drawdown"]]

,strategy,terminal_mean,terminal_median,probability_of_loss,probability_of_ruin,expected_maximum_drawdown
0,flat,1660.333333,1640.000000,0.160000,0.000000,0.244372
1,martingale,1856.333333,450.000000,0.580000,0.000000,0.415898
2,reverse_martingale,1743.733333,2010.000000,0.133333,0.000000,0.237126
3,full_kelly,1858.366667,2130.500000,0.386667,0.000000,0.374984
4,half_kelly,1830.066667,2122.500000,0.160000,0.000000,0.328062
5,quarter_kelly,1734.713333,1873.500000,0.113333,0.000000,0.208312


## Limitations

- The biased CSV is synthetic and demonstrates workflow, not a claim about a live casino wheel.
- Bias selection and evaluation should use disjoint samples.
- Chi-square asymptotics require adequate expected counts; the project also reports a multinomial Monte Carlo result.
- Kelly is conditional log-growth optimisation under a correct probability and payout model. It is not guaranteed profit.
- Finite simulation summaries carry Monte Carlo error and depend on stop rules, table limits and the random seed.